# 공정변수와 생산성과의 상관분석

정상 배치 90개를 배치 단위로 요약한 뒤 공정변수와 성과지표의 Spearman 상관을 주 분석, Pearson 상관을 보조 분석으로 계산한다. 65개 조합의 p값에는 FDR 보정을 적용한다. 상관은 인과관계가 아니다.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats


def find_project_root(start=Path.cwd()):
    for root in (start, *start.parents):
        if (root / 'data/interim/merged_data_ko.csv').exists():
            return root
    raise FileNotFoundError('merged_data_ko.csv를 찾을 수 없습니다.')


def fdr_bh(p_values):
    p_values = np.asarray(p_values, dtype=float)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adjusted = ranked * len(ranked) / np.arange(1, len(ranked) + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    result = np.empty_like(adjusted)
    result[order] = np.clip(adjusted, 0, 1)
    return result


data = pd.read_csv(find_project_root() / 'data/interim/merged_data_ko.csv')
data = data.loc[data['배치번호'] <= 90].copy()
print(f'정상 배치 {data["배치번호"].nunique()}개')

정상 배치 90개


### 확인

Fault의 극단적인 공정 패턴이 상관계수를 지배하지 않도록 정상 배치 1~90만 사용한다.

In [2]:
def slope(x, y):
    return float(np.polyfit(x, y, 1)[0])


rows = []
for batch_number, batch in data.groupby('배치번호', sort=True):
    batch = batch.sort_values('발효시간(h)')
    time = batch['발효시간(h)'].to_numpy()
    end_time = time[-1]
    late = time / end_time >= 0.8
    penicillin = batch['페니실린농도(g/L)'].to_numpy()
    substrate = batch['기질농도(g/L)'].to_numpy()
    our = batch['산소소모율(g/min)'].to_numpy()
    rows.append({
        '배치번호': batch_number, '최종농도': penicillin[-1],
        '총수확량': batch['총수확량(kg)'].iloc[0],
        '농도유지율': penicillin[-1] / penicillin.max(),
        '생산성': penicillin[-1] / end_time,
        '후기농도기울기': slope(time[late], penicillin[late]),
        'pH표준편차': batch['pH'].std(ddof=1),
        'DO최솟값': batch['용존산소(mg/L)'].min(), 'DO평균': batch['용존산소(mg/L)'].mean(),
        'OUR평균': our.mean(), 'OUR음수비율': np.mean(our < 0),
        'CO2평균': batch['배가스이산화탄소(%)'].mean(), 'CO2최댓값': batch['배가스이산화탄소(%)'].max(),
        '기질최종': substrate[-1], '기질후기기울기': slope(time[late], substrate[late]),
        '온도표준편차': batch['발효온도(K)'].std(ddof=1),
        '산총투입량': np.trapezoid(batch['산투입유량(L/h)'], time),
        '염기총투입량': np.trapezoid(batch['염기투입유량(L/h)'], time),
        '당총공급량': np.trapezoid(batch['당공급유량(L/h)'], time),
    })
batch_metrics = pd.DataFrame(rows)
display(batch_metrics.head())

,배치번호,최종농도,총수확량,농도유지율,생산성,후기농도기울기,pH표준편차,DO최솟값,DO평균,OUR평균,OUR음수비율,CO2평균,CO2최댓값,기질최종,기질후기기울기,온도표준편차,산총투입량,염기총투입량,당총공급량
0,1,29.373,2786400.0,0.998776,0.129969,0.074064,0.018110,9.3220,12.449236,1.327662,0.007080,1.468085,1.9004,0.001474,0.000004,0.086883,2.804832,15614.073620,17286.2
1,2,30.416,2326000.0,1.000000,0.132243,0.063440,0.023937,9.1854,12.774579,1.375537,0.006957,1.432999,2.0410,0.001862,0.000020,0.142420,1.455092,15898.861536,17606.2
2,3,17.428,2675300.0,0.620147,0.062691,-0.102661,0.030914,6.3812,11.021012,1.291738,0.002158,1.440264,2.1591,56.757000,0.691102,0.146234,46.514560,15715.107278,21446.2
3,4,15.107,1886700.0,0.636352,0.065683,-0.113900,0.029015,7.6010,12.283719,1.045671,0.006087,1.332977,1.9361,42.947000,0.642771,0.100269,32.187242,11316.673026,17606.2
4,5,28.172,3562900.0,1.000000,0.157385,0.128194,0.019718,8.8033,12.050972,1.465969,0.008939,1.467429,2.1286,0.001331,0.000028,0.099885,1.765646,11395.607916,13526.2


### 확인

각 배치가 한 행이므로 동일 배치의 반복 측정값을 독립 표본처럼 사용하는 의사 반복을 피했다.

In [3]:
features = ['pH표준편차', 'DO최솟값', 'DO평균', 'OUR평균', 'OUR음수비율', 'CO2평균', 'CO2최댓값',
            '기질최종', '기질후기기울기', '온도표준편차', '산총투입량', '염기총투입량', '당총공급량']
outcomes = ['최종농도', '총수확량', '농도유지율', '생산성', '후기농도기울기']
rows = []
for feature in features:
    for outcome in outcomes:
        spearman = stats.spearmanr(batch_metrics[feature], batch_metrics[outcome])
        pearson = stats.pearsonr(batch_metrics[feature], batch_metrics[outcome])
        rows.append({'공정변수': feature, '성과': outcome,
                     'Spearman_rho': spearman.statistic, 'Spearman_p': spearman.pvalue,
                     'Pearson_r': pearson.statistic, 'Pearson_p': pearson.pvalue})
correlation_results = pd.DataFrame(rows)
correlation_results['Spearman_FDR'] = fdr_bh(correlation_results['Spearman_p'])
correlation_results['Pearson_FDR'] = fdr_bh(correlation_results['Pearson_p'])
significant_correlations = correlation_results.loc[correlation_results['Spearman_FDR'] < 0.05].copy()
significant_correlations = significant_correlations.sort_values('Spearman_rho', key=lambda x: x.abs(), ascending=False)
print(f'Spearman FDR 유의: {len(significant_correlations)}/{len(correlation_results)}개')
display(significant_correlations.round(6))

Spearman FDR 유의: 36/65개


,공정변수,성과,Spearman_rho,Spearman_p,Pearson_r,Pearson_p,Spearman_FDR,Pearson_FDR
53,산총투입량,생산성,-0.831864,0.000000,-0.939783,0.000000,0.000000,0.000000
52,산총투입량,농도유지율,-0.821898,0.000000,-0.972743,0.000000,0.000000,0.000000
37,기질최종,농도유지율,-0.820123,0.000000,-0.974856,0.000000,0.000000,0.000000
50,산총투입량,최종농도,-0.815401,0.000000,-0.929595,0.000000,0.000000,0.000000
42,기질후기기울기,농도유지율,-0.763932,0.000000,-0.934815,0.000000,0.000000,0.000000
0,pH표준편차,최종농도,-0.735885,0.000000,-0.090484,0.396354,0.000000,0.495442
54,산총투입량,후기농도기울기,-0.728493,0.000000,-0.781658,0.000000,0.000000,0.000000
38,기질최종,생산성,-0.722007,0.000000,-0.928865,0.000000,0.000000,0.000000
39,기질최종,후기농도기울기,-0.720739,0.000000,-0.762803,0.000000,0.000000,0.000000
40,기질후기기울기,최종농도,-0.703996,0.000000,-0.885031,0.000000,0.000000,0.000000


### 판단

- 65개 조합 중 Spearman FDR 기준 36개가 유의했다. 상관 구조가 강하므로 변수 선택 시 다중공선성과 중복 정보를 확인해야 한다.
- 가장 강한 관계는 산 총투입량과 생산성(ρ=-0.832), 산 총투입량과 농도유지율(ρ=-0.822), 기질 최종값과 농도유지율(ρ=-0.820)이다.
- 기질 최종값·후기 기질 기울기가 클수록 최종농도, 유지율, 생산성이 낮은 경향이 뚜렷하다. 후기 기질 축적은 저성과 후보 신호다.
- OUR 평균은 최종농도(ρ=0.520), 유지율(ρ=0.574), 생산성(ρ=0.570)과 양의 관계다.
- pH 표준편차는 최종농도와 Spearman ρ=-0.736이지만 Pearson은 유의하지 않았다. 비선형성 또는 일부 배치 영향 가능성이 있어 산점도와 배치 궤적 확인이 필요하다.
- 이 결과는 공정변수의 원인 효과를 증명하지 않는다. 전략별 운전 방식과 종료시간이 함께 영향을 줄 수 있으므로 회귀·교차검증에서 재확인한다.